**Tema:** funciones

---

## Sección 6. Funciones

Una función agrupa un bloque de código con un nombre para poder **reutilizarlo**. Python trae muchas funciones integradas (`len()`, `sorted()`, `max()`…) y permite definir las nuestras con `def`. En ingeniería de datos, cada paso de un pipeline (leer, validar, transformar, cargar) suele ser una función. Así cada paso se puede probar por separado y combinar con otros.

```python
def nombre_funcion(parametros):
    """Docstring: qué hace la función."""
    # cuerpo de la función
    return resultado
```

Por convención, los nombres de función van en minúsculas y con las palabras separadas por `_`: `calcular_kpis()`, `validar_registro()`.

### Ejemplos resueltos

#### Ejemplo 6.1 · Devolver varios valores

In [0]:
def resumen_lote(valores):
    """Devuelve (n, mínimo, máximo, media) de una lista de números."""
    n = len(valores)
    media = sum(valores) / n
    return n, min(valores), max(valores), round(media, 2)

lote_lunes = [120.0, 85.5, 430.0, 95.25]
lote_martes = [300.0, 310.5]

print(resumen_lote(lote_lunes))
n, minimo, maximo, media = resumen_lote(lote_martes)
print(f"Martes:{n} registros, media{media}")

(4, 85.5, 430.0, 182.69)
Martes:2 registros, media305.25


**Paso a paso**

1. El texto entre triples comillas es el *docstring*: documenta qué hace la función.
2. `return n, min(...), ...` devuelve **una tupla**. El que llama puede guardarla entera o desempaquetarla en variables.
3. La misma función sirve para cualquier lote. Esa es la ventaja de definirla una vez.

#### Ejemplo 6.2 · Valores por defecto y argumentos por nombre

In [0]:
def normalizar(texto, minusculas=True, quitar_espacios=True, relleno=""):
    if texto is None or texto.strip() == "":
        return relleno
    if quitar_espacios:
        texto = texto.strip()
    if minusculas:
        texto = texto.lower()
    return texto

print(repr(normalizar("  Madrid ")))
print(repr(normalizar("  Madrid ", minusculas=False)))
print(repr(normalizar("", relleno="SIN_DATO")))
print(repr(normalizar(None, relleno="SIN_DATO")))
print(repr(normalizar("  BCN  ", quitar_espacios=False)))

'madrid'
'Madrid'
'SIN_DATO'
'SIN_DATO'
'  bcn  '


**Paso a paso**

1. `minusculas`, `quitar_espacios` y `relleno` tienen **valor por defecto**. Si no se pasan, se usan esos valores.
2. Al llamar con `minusculas=False` (**argumento por nombre**) cambiamos solo esa opción, sin tener que pasar las anteriores en orden.
3. En la definición, los parámetros con valor por defecto deben ir **después** de los obligatorios.
4. `repr()` muestra el texto entre comillas, así que se ven los espacios que quedan.

#### Ejemplo 6.3 · Reglas de llamada: posición, nombre, `None` y redefinición

In [0]:
def publicar_metrica(nombre, valor, unidad):
    print(f"métrica={nombre} valor={valor} unidad={unidad}")

publicar_metrica("filas_cargadas", 15230, "filas")            # por posición
publicar_metrica(valor=42.7, unidad="s", nombre="duracion")   # por nombre: el orden da igual
publicar_metrica("errores", unidad="filas", valor=3)          # mezcla: posicionales primero

# publicar_metrica(nombre="errores", 3, "filas")
#   -> SyntaxError: positional argument follows keyword argument
# publicar_metrica("errores", 3)
#   -> TypeError: publicar_metrica() missing 1 required positional argument: 'unidad'

resultado = publicar_metrica("latencia_p95", 230, "ms")
print("La función devuelve:", resultado)

def formato_fecha():
    return "AAAA-MM-DD"

def formato_fecha():           # redefinición: sustituye a la anterior
    return "DD/MM/AAAA"

print(formato_fecha())

métrica=filas_cargadas valor=15230 unidad=filas
métrica=duracion valor=42.7 unidad=s
métrica=errores valor=3 unidad=filas
métrica=latencia_p95 valor=230 unidad=ms
La función devuelve: None
DD/MM/AAAA


**Paso a paso**

1. **Por posición**, cada valor va al parámetro que ocupa su mismo lugar. Hay que pasar exactamente tantos valores como parámetros obligatorios tenga la función.
2. **Por nombre**, cada valor se asigna al parámetro indicado y el orden da igual.
3. Se pueden mezclar, pero los argumentos posicionales siempre van **antes** que los argumentos por nombre. Si no, Python ni siquiera ejecuta el programa (`SyntaxError`).
4. Si falta un argumento obligatorio, la llamada falla con `TypeError` e indica cuál falta.
5. Una función sin `return` **devuelve `None`**. `publicar_metrica()` imprime, pero no devuelve nada.
6. Si se define dos veces una función con el mismo nombre, la segunda definición sustituye a la primera. En un notebook es una fuente habitual de confusión: se ejecuta la versión de la última celda que se ejecutó.


#### Ejemplo 6.4 · Número variable de argumentos: `args` y `*kwargs`

In [0]:
def total_bytes(*tamanos):
    print("  recibido:", tamanos)
    return sum(tamanos)

print(total_bytes(1024, 2048))
print(total_bytes(500, 500, 500, 500))
print(total_bytes())

def log_evento(nivel, mensaje, **contexto):
    linea = f"[{nivel}]{mensaje}"
    for clave, valor in contexto.items():
        linea += f" |{clave}={valor}"
    print(linea)

log_evento("INFO", "Carga finalizada", tabla="ventas", filas=15230)
log_evento("ERROR", "Timeout en origen", origen="api_crm", intento=3, espera_s=30)
log_evento("DEBUG", "Sin contexto")

  recibido: (1024, 2048)
3072
  recibido: (500, 500, 500, 500)
2000
  recibido: ()
0
[INFO]Carga finalizada |tabla=ventas |filas=15230
[ERROR]Timeout en origen |origen=api_crm |intento=3 |espera_s=30
[DEBUG]Sin contexto


**Paso a paso**

1. `tamanos` recoge todos los argumentos posicionales en una **tupla**, que puede estar vacía.
2. `*contexto` recoge todos los argumentos por nombre que sobran en un **diccionario**. Es muy útil para logs, porque cada evento puede llevar un contexto distinto.
3. Los nombres `args` y `kwargs` son solo una convención: lo que importa son el  y el `*`.

#### Ejemplo 6.5 · La firma completa de una función

Cuando una función combina todos los tipos de parámetros, deben aparecer en este orden:

| Orden | Tipo | En el ejemplo |
| --- | --- | --- |
| 1 | Posicionales obligatorios | `nombre`, `entorno` |
| 2 | Posicionales variables | `*tablas` |
| 3 | Solo por nombre | `reintentos`, `notificar` |
| 4 | Por nombre variables | `**spark_conf` |

In [0]:
def lanzar_job(nombre, entorno, *tablas, reintentos=3, notificar=False, **spark_conf):
    print(f"Job:{nombre} [{entorno}]")
    print("  Tablas:", tablas)
    print("  Reintentos:", reintentos, "| Notificar:", notificar)
    print("  Configuración Spark:", spark_conf)

lanzar_job("carga_diaria", "prod", "ventas", "clientes", "stock",
           notificar=True, executor_memory="8g", shuffle_partitions=200)

lanzar_job("recalculo_kpis", "dev")

# Cuidado: el 5 no llega a reintentos, se queda dentro de tablas
lanzar_job("backfill", "dev", "ventas", 5)

Job:carga_diaria [prod]
  Tablas: ('ventas', 'clientes', 'stock')
  Reintentos: 3 | Notificar: True
  Configuración Spark: {'executor_memory': '8g', 'shuffle_partitions': 200}
Job:recalculo_kpis [dev]
  Tablas: ()
  Reintentos: 3 | Notificar: False
  Configuración Spark: {}
Job:backfill [dev]
  Tablas: ('ventas', 5)
  Reintentos: 3 | Notificar: False
  Configuración Spark: {}


**Paso a paso**

1. `nombre` y `entorno` reciben los dos primeros valores. Todos los posicionales que siguen van a la tupla `tablas`.
2. Los parámetros escritos **después** de `tablas` solo se pueden pasar **por nombre**. Por eso `notificar=True` llega a su parámetro.
3. Los argumentos por nombre que no corresponden a ningún parámetro (`executor_memory`, `shuffle_partitions`) van al diccionario `spark_conf`.
4. En la última llamada, el `5` no llega a `reintentos`: al ser posicional, se queda dentro de `tablas`. Para cambiar los reintentos hay que escribir `reintentos=5`.

#### Ejemplo 6.6 · Pasar argumentos desde una tupla o un diccionario

Es habitual que los parámetros de conexión estén en un fichero de configuración. Al leerlo, llegan a Python como tupla o como diccionario.

In [0]:
def cadena_conexion(host, puerto, base_datos, usuario="lector"):
    return f"{usuario}@{host}:{puerto}/{base_datos}"

params_tupla = ("db-prod.interno", 5432, "ventas")
print(cadena_conexion(*params_tupla))

params_dict = {"host": "db-dev.interno", "puerto": 5433,
               "base_datos": "pruebas", "usuario": "etl"}
print(cadena_conexion(**params_dict))

print(cadena_conexion("db-qa.interno", 5432, base_datos="qa"))

# Sin ** se desempaquetan las CLAVES del diccionario, no los valores
print(cadena_conexion(*params_dict))

lector@db-prod.interno:5432/ventas
etl@db-dev.interno:5433/pruebas
lector@db-qa.interno:5432/qa
usuario@host:puerto/base_datos


**Paso a paso**

1. `params_tupla` reparte los elementos de la tupla como argumentos posicionales, en orden.
2. `*params_dict` reparte el diccionario como argumentos por nombre. Las claves deben coincidir con los nombres de los parámetros.
3. En la tercera llamada se mezclan argumentos posicionales y por nombre. Los posicionales siempre van primero.
4. Con un solo , un diccionario reparte sus **claves** como argumentos posicionales. Por eso la última línea muestra los nombres de los campos en lugar de sus valores.

#### Ejemplo 6.8 · Funciones internas

Una función se puede definir dentro de otra. La función interna solo existe mientras se ejecuta la externa, así que queda protegida: nadie puede llamarla desde fuera por error.

In [0]:
def validar_lote(registros, importe_maximo=10000):
    def es_importe_valido(importe):        # función interna
        return importe is not None and 0 <= importe <= importe_maximo

    validos = 0
    for r in registros:
        if es_importe_valido(r["importe"]):
            validos += 1
    return validos, len(registros) - validos

lote = [
    {"id": 1, "importe": 250.0},
    {"id": 2, "importe": -4.0},
    {"id": 3, "importe": None},
    {"id": 4, "importe": 12500.0},
    {"id": 5, "importe": 99.9},
]
print("(válidos, rechazados):", validar_lote(lote))
print("Con máximo 20000:     ", validar_lote(lote, importe_maximo=20000))

# es_importe_valido(100)  -> NameError: name 'es_importe_valido' is not defined

(válidos, rechazados): (2, 3)
Con máximo 20000:      (3, 2)


**Paso a paso**

1. `es_importe_valido()` se define dentro de `validar_lote()`. Fuera de ella no existe: intentar llamarla produce `NameError`.
2. La función interna puede **leer** las variables de la externa. Usa `importe_maximo` sin recibirlo como parámetro.
3. Por eso, al llamar a `validar_lote()` con `importe_maximo=20000`, la regla interna cambia y el importe de 12500 pasa a ser válido.
4. Es una buena opción para reglas auxiliares que solo tienen sentido dentro de un proceso concreto.

#### Ejemplo 6.9 · Un mini pipeline hecho con funciones

In [0]:
def leer():
    return ["  101;Acme;1200 ", "102;Globex;", "103;initech;300", "  104;ACME;50"]

def parsear(linea):
    id_, cliente, importe = linea.strip().split(";")
    return {
        "id": int(id_),
        "cliente": cliente.title(),
        "importe": float(importe) if importe else None,
    }

def es_valido(registro):
    return registro["importe"] is not None

def ejecutar_pipeline():
    validos, rechazados = [], []
    for linea in leer():
        registro = parsear(linea)
        if es_valido(registro):
            validos.append(registro)
        else:
            rechazados.append(registro)
    return validos, rechazados

ok, ko = ejecutar_pipeline()
print("Válidos:", ok)
print("Rechazados:", ko)
print("Total cargado:", sum([r["importe"] for r in ok]))

Válidos: [{'id': 101, 'cliente': 'Acme', 'importe': 1200.0}, {'id': 103, 'cliente': 'Initech', 'importe': 300.0}, {'id': 104, 'cliente': 'Acme', 'importe': 50.0}]
Rechazados: [{'id': 102, 'cliente': 'Globex', 'importe': None}]
Total cargado: 1550.0


**Paso a paso**

1. Cada función hace **una sola cosa**: `leer()` obtiene los datos, `parsear()` convierte una línea en un diccionario, `es_valido()` aplica una regla de calidad y `ejecutar_pipeline()` coordina el proceso.
2. `parsear()` normaliza el nombre del cliente con `title()`, de modo que `ACME` e `initech` pasan a `Acme` e `Initech`.
3. Las filas sin importe no se pierden: se apartan en `rechazados`, igual que una tabla de cuarentena.


#### Ejemplo 6.10 · Un job MapReduce organizado en funciones

En Hadoop, el programador solo escribe dos funciones, la de **map** y la de **reduce**, y el framework se encarga de ejecutarlas en paralelo y de mover los datos entre ellas. Vamos a simular un conteo de las palabras más buscadas en una tienda online.

In [0]:
def mapear(linea):
    """Fase map: emite (palabra, 1) por cada palabra buscada."""
    usuario, termino = linea.split(";")
    return [(palabra.lower(), 1) for palabra in termino.split()]

def agrupar(pares):
    """Fase shuffle: reúne todos los valores de cada clave."""
    grupos = {}
    for clave, valor in pares:
        grupos.setdefault(clave, []).append(valor)
    return grupos

def reducir(clave, valores):
    """Fase reduce: combina los valores de una clave."""
    return clave, sum(valores)

def ejecutar_job(lineas):
    pares = []
    for linea in lineas:
        pares.extend(mapear(linea))
    grupos = agrupar(pares)
    return [reducir(clave, valores) for clave, valores in grupos.items()]

busquedas = [
    "u1;Zapatillas running", "u2;zapatillas mujer", "u3;Camiseta running",
    "u1;zapatillas trail", "u4;mochila trail", "u5;Running",
]

print("Map de una línea:", mapear(busquedas[0]))
resultado = ejecutar_job(busquedas)
from operator import itemgetter
ordenado = sorted(sorted(resultado), key=itemgetter(1), reverse=True)
for palabra, total in ordenado:
    print(f"{palabra:<11}{total}")

Map de una línea: [('zapatillas', 1), ('running', 1)]
running    3
zapatillas 3
trail      2
camiseta   1
mochila    1
mujer      1


**Paso a paso**

1. `mapear()` procesa **una sola línea** y devuelve sus pares `(palabra, 1)`. Como no depende de ninguna otra línea, el framework puede ejecutarla en muchos nodos a la vez.
2. `agrupar()` simula el **shuffle**: reúne en una lista todos los valores de una misma clave.
3. `reducir()` recibe **una clave y sus valores** y los combina en un resultado.
4. `ejecutar_job()` hace el papel del framework: llama a las tres fases en orden.
5. Para ordenar por frecuencia y, en caso de empate, alfabéticamente, se ordena primero por palabra y después por total. `sorted()` respeta el orden previo de los empates.

#### Ejemplo 6.11 · Calcular el número de particiones de un dataset

En Spark, una referencia habitual es que cada partición pese unos 128 MB. Con pocas particiones se desaprovechan núcleos del clúster; con demasiadas, se pierde tiempo gestionando tareas muy pequeñas.

In [0]:
import math

def calcular_particiones(tamano_gb, objetivo_mb=128, nucleos=None):
    """Número de particiones para que cada una pese unos objetivo_mb."""
    particiones = math.ceil(tamano_gb * 1024 / objetivo_mb)
    if nucleos and particiones > nucleos:
        # redondear hacia arriba a un múltiplo del número de núcleos
        particiones = math.ceil(particiones / nucleos) * nucleos
    return particiones

datasets = [("logs_web", 50), ("ventas", 2.5), ("clientes", 0.04)]

for nombre, gb in datasets:
    simple = calcular_particiones(gb)
    ajustado = calcular_particiones(gb, nucleos=48)
    print(f"{nombre:<9}{gb:>5} GB ->{simple:>3} particiones | con 48 núcleos:{ajustado}")

print("logs_web con particiones de 256 MB:", calcular_particiones(50, objetivo_mb=256))

logs_web    50 GB ->400 particiones | con 48 núcleos:432
ventas     2.5 GB -> 20 particiones | con 48 núcleos:20
clientes  0.04 GB ->  1 particiones | con 48 núcleos:1
logs_web con particiones de 256 MB: 200


**Paso a paso**

1. `math.ceil()` redondea hacia arriba: si el dataset pesa 40 MB, hace falta al menos una partición.
2. `objetivo_mb` y `nucleos` tienen valores por defecto, así que la función sirve tanto para un cálculo rápido como para uno ajustado al clúster.
3. Si se indican núcleos y hay más particiones que núcleos, el resultado se redondea a un múltiplo de los núcleos. Así la última “oleada” de tareas ocupa el clúster entero: 400 particiones en 48 núcleos dejarían una última oleada a medias, y 432 no.
4. Si hay menos particiones que núcleos (`ventas`, `clientes`), no se ajusta nada: crear particiones casi vacías no aporta.

#### Ejemplo 6.12 · Construir prompts para un LLM con una función

En las aplicaciones de IA Generativa, los prompts no se escriben a mano cada vez: se generan con funciones que combinan una plantilla fija con los datos de cada petición.

In [0]:
def construir_prompt(tarea, contexto, *ejemplos, idioma="español",
                     formato="lista", **restricciones):
    partes = [f"Tarea:{tarea}", f"Contexto:{contexto}"]
    if ejemplos:
        partes.append("Ejemplos:")
        for i, ejemplo in enumerate(ejemplos, start=1):
            partes.append(f"{i}.{ejemplo}")
    partes.append(f"Responde en{idioma} con formato{formato}.")
    for clave, valor in restricciones.items():
        partes.append(f"-{clave.replace('_', ' ')}:{valor}")
    return "\n".join(partes)

prompt = construir_prompt(
    "Resumir las incidencias del pipeline nocturno",
    "3 jobs fallaron por timeout en la API del CRM y 1 job tardó el doble de lo habitual.",
    "job_ventas: timeout tras 3 reintentos -> revisar el límite de la API",
    longitud_maxima="5 líneas",
    tono="técnico",
)
print(prompt)
print()
print(construir_prompt("Traducir el mensaje de error", "Connection refused", idioma="inglés",
                       formato="texto plano"))

Tarea:Resumir las incidencias del pipeline nocturno
Contexto:3 jobs fallaron por timeout en la API del CRM y 1 job tardó el doble de lo habitual.
Ejemplos:
1.job_ventas: timeout tras 3 reintentos -> revisar el límite de la API
Responde enespañol con formatolista.
-longitud maxima:5 líneas
-tono:técnico

Tarea:Traducir el mensaje de error
Contexto:Connection refused
Responde eninglés con formatotexto plano.


**Paso a paso**

1. `tarea` y `contexto` son obligatorios: sin ellos el prompt no tiene sentido.
2. `ejemplos` admite cero o más ejemplos (*few-shot*). Si no se pasa ninguno, la sección de ejemplos no aparece.
3. `idioma` y `formato` tienen un valor por defecto y solo se pueden pasar por nombre.
4. `*restricciones` admite cualquier instrucción adicional. El nombre del argumento se convierte en texto sustituyendo `_` por espacios.
5. `"\n".join(partes)` une las líneas con saltos de línea.

#### Ejemplo 6.13 · Dividir documentos en *chunks* para un sistema RAG

En un sistema RAG (*Retrieval-Augmented Generation*), los documentos se trocean en fragmentos (*chunks*) antes de convertirlos en embeddings. Los fragmentos se **solapan** un poco para que una idea no quede partida entre dos trozos sin contexto.

In [0]:
def dividir_en_chunks(texto, tamano=8, solape=2):
    palabras = texto.split()
    paso = tamano - solape
    chunks = []
    for inicio in range(0, len(palabras), paso):
        trozo = palabras[inicio:inicio + tamano]
        chunks.append({"id": len(chunks), "inicio": inicio, "texto": " ".join(trozo)})
        if inicio + tamano >= len(palabras):
            break
    return chunks

politica = ("La política de devoluciones permite devolver cualquier producto en un plazo "
            "de 30 días. El cliente debe conservar el ticket de compra. Los productos de "
            "higiene no admiten devolución una vez abiertos.")

print("Palabras:", len(politica.split()))
for chunk in dividir_en_chunks(politica):
    print(chunk)
print("Chunks con tamano=12 y solape=3:", len(dividir_en_chunks(politica, tamano=12, solape=3)))

Palabras: 32
{'id': 0, 'inicio': 0, 'texto': 'La política de devoluciones permite devolver cualquier producto'}
{'id': 1, 'inicio': 6, 'texto': 'cualquier producto en un plazo de 30 días.'}
{'id': 2, 'inicio': 12, 'texto': '30 días. El cliente debe conservar el ticket'}
{'id': 3, 'inicio': 18, 'texto': 'el ticket de compra. Los productos de higiene'}
{'id': 4, 'inicio': 24, 'texto': 'de higiene no admiten devolución una vez abiertos.'}
Chunks con tamano=12 y solape=3: 4


**Paso a paso**

1. Cada chunk tiene `tamano` palabras, y el siguiente empieza `tamano - solape` palabras más adelante. Con 8 y 2, avanza de 6 en 6, así que cada chunk repite las 2 últimas palabras del anterior.
2. Cada chunk se guarda como un diccionario con su `id` y su posición de `inicio`. Esos metadatos permiten saber después de qué parte del documento salió cada fragmento.
3. El `break` detiene el bucle cuando un chunk ya llega al final del texto, para no generar un último fragmento formado solo por el solape.
4. Los sistemas reales trocean por **tokens** en lugar de palabras, pero la lógica es la misma.

### Ejercicios propuestos

#### Ejercicio 6.1 · Función de validación de registros
Escribe `validar_registro(registro, obligatorios)`, que devuelva una tupla `(es_valido, lista_de_errores)`. Un campo obligatorio falla si no existe, si es `None` o si es una cadena vacía. Además, si el importe existe y es negativo, se añade el error `"importe negativo"`. Aplica la función a todos los registros y muestra `?` cuando falte el id.

In [0]:
registros = [
    {"id": 1, "cliente": "Acme", "importe": 120.0},
    {"id": 2, "cliente": "", "importe": 80.0},
    {"id": 3, "cliente": "Globex", "importe": -15.0},
    {"cliente": "Initech", "importe": None},
]
obligatorios = ["id", "cliente", "importe"]


def validar_registro(registro, obligatorios):
    errores = []
    for campo in obligatorios:
        valor = registro.get(campo)
        if valor is None or valor == "":
            errores.append(f"falta '{campo}'")

    if "importe" in registro and registro["importe"] is not None:
        if registro["importe"] < 0:
            errores.append("importe negativo")

    es_valido = len(errores) == 0
    return es_valido, errores


for reg in registros:
    es_valido, errores = validar_registro(reg, obligatorios)
    estado = "OK" if es_valido else "KO"
    id_mostrar = reg.get("id", "?")
    print(f"{estado}  {id_mostrar} {errores}")

OK  1 []
KO  2 ["falta 'cliente'"]
KO  3 ['importe negativo']
KO  ? ["falta 'id'", "falta 'importe'"]


#### Ejercicio 6.2 · KPIs con argumentos variables

Escribe `calcular_kpis(*importes, moneda="EUR", decimales=2)`, que devuelva un diccionario con `n`, `total`, `media` y `moneda`. Si no recibe importes, la media debe ser `None`. Comprueba la función con estas llamadas:

In [0]:
def calcular_kpis(*importes, moneda="EUR", decimales=2):
    n = len(importes)
    if n == 0:
        return {"n": 0, "total": 0, "media": None, "moneda": moneda}

    total = sum(importes)
    media = total / n

    if decimales > 0:
        total = round(total, decimales)
        media = round(media, decimales)
    else:
        total = int(round(total))
        media = round(media, 1)

    return {"n": n, "total": total, "media": media, "moneda": moneda}


print(calcular_kpis(120.5, 80.25, 99.99))
print(calcular_kpis(1000, 2500, moneda="USD", decimales=0))
print(calcular_kpis())
lote = [10.0, 20.0, 30.0]
print(calcular_kpis(*lote, moneda="MXN"))

{'n': 3, 'total': 300.74, 'media': 100.25, 'moneda': 'EUR'}
{'n': 2, 'total': 3500, 'media': 1750.0, 'moneda': 'USD'}
{'n': 0, 'total': 0, 'media': None, 'moneda': 'EUR'}
{'n': 3, 'total': 60.0, 'media': 20.0, 'moneda': 'MXN'}


#### Ejercicio 6.3 · Construir rutas de un data lake

Escribe `construir_ruta(base, *carpetas, **particiones)`. La función debe unir la base (sin `/` final), las carpetas y las particiones con el formato `clave=valor`, separando cada parte con `/`. Comprueba la función con estas llamadas:

In [0]:
def construir_ruta(base, *carpetas, **particiones):
    partes = [base.rstrip("/")]
    partes.extend(carpetas)
    partes.extend(f"{clave}={valor}" for clave, valor in particiones.items())
    return "/".join(partes)


print(
    construir_ruta(
        "abfss://lake@cuenta.dfs.core.windows.net/",
        "plata",
        "ventas",
        year=2026,
        month="10",
        day="15",
    )
)
print(construir_ruta("/lake", "bronce", "clientes"))
print(construir_ruta("/lake", region="EU"))

abfss://lake@cuenta.dfs.core.windows.net/plata/ventas/year=2026/month=10/day=15
/lake/bronce/clientes
/lake/region=EU


#### Ejercicio 6.4 · Una función de conteo reutilizable
Escribe `contar_por(registros, campo, ausente="SIN_VALOR")`, que cuente cuántos registros hay para cada valor del campo indicado. Los registros que no tengan el campo se cuentan con el valor de `ausente`. Llama a la función tres veces: por `canal`, por `pais` y por `pais` con `ausente="??"`.

In [0]:
pedidos = [
    {"id": 1, "canal": "web", "pais": "ES"},
    {"id": 2, "canal": "app", "pais": "PT"},
    {"id": 3, "canal": "web", "pais": "ES"},
    {"id": 4, "canal": "tienda", "pais": "ES"},
    {"id": 5, "canal": "web", "pais": "MX"},
    {"id": 6, "canal": "app"},
]


def contar_por(registros, campo, ausente="SIN_VALOR"):
    conteo = {}
    for r in registros:
        valor = r.get(campo, ausente)
        conteo[valor] = conteo.get(valor, 0) + 1
    return conteo


print(contar_por(pedidos, "canal"))
print(contar_por(pedidos, "pais"))
print(contar_por(pedidos, "pais", ausente="??"))

{'web': 3, 'app': 2, 'tienda': 1}
{'ES': 3, 'PT': 1, 'MX': 1, 'SIN_VALOR': 1}
{'ES': 3, 'PT': 1, 'MX': 1, '??': 1}


#### Ejercicio 6.5 · Organizar un script en funciones

Tienes estas líneas de un fichero de movimientos por tienda: `fecha;tienda;tipo;importe`.
Organiza la solución en cuatro funciones:

- `parsear_linea(linea)`: devuelve un diccionario. El tipo se normaliza quitando espacios y pasando a minúsculas. Si no hay importe, se guarda `None`.
- `es_valida(registro)`: comprueba que haya importe.
- `agregar_por_tienda(registros, tipo)`: devuelve un diccionario `tienda → total` con los registros del tipo indicado.
- `informe(lineas)`: coordina el proceso y muestra cuántas líneas se han leído, cuántas son válidas y cuántas se han rechazado, seguido del total de `ventas` por tienda, ordenado.

In [0]:
lineas_crudas = [
    "2026-10-15;MAD01;  ventas ;1250.0",
    "2026-10-15;BCN02;Ventas;",
    "2026-10-15;MAD01;DEVOLUCIONES;-80.0",
    "2026-10-15;VLC03;ventas;410.0",
    "2026-10-15;BCN02;ventas;980.5",
]


def parsear_linea(linea):
    partes = linea.split(";")
    fecha = partes[0]
    tienda = partes[1]
    tipo = partes[2].strip().lower()
    importe_str = partes[3].strip()
    importe = float(importe_str) if importe_str else None
    return {"fecha": fecha, "tienda": tienda, "tipo": tipo, "importe": importe}


def es_valida(registro):
    return registro["importe"] is not None


def agregar_por_tienda(registros, tipo):
    totales = {}
    for r in registros:
        if r["tipo"] == tipo:
            tienda = r["tienda"]
            totales[tienda] = totales.get(tienda, 0.0) + r["importe"]
    return totales


def informe(lineas):
    registros = [parsear_linea(l) for l in lineas]
    validos = [r for r in registros if es_valida(r)]
    leidas = len(registros)
    n_validas = len(validos)
    rechazadas = leidas - n_validas

    print(f"Leídas: {leidas} | Válidas: {n_validas} | Rechazadas: {rechazadas}")

    totales_ventas = agregar_por_tienda(validos, "ventas")
    for tienda in sorted(totales_ventas):
        print(f"  {tienda}: {totales_ventas[tienda]:.2f}")


informe(lineas_crudas)

Leídas: 5 | Válidas: 4 | Rechazadas: 1
  BCN02: 980.50
  MAD01: 1250.00
  VLC03: 410.00


#### Ejercicio 6.6 · ¿Qué llamadas fallan?

Sin ejecutar el código, analiza estas seis llamadas a la misma función:
1. Indica qué llamadas producen un error y explica por qué.
2. Para las que funcionan, escribe qué imprimen.
3. Una de las llamadas que funcionan tiene un error lógico: hace algo distinto de lo que probablemente pretendía quien la escribió. Identifícala y corrígela.

Después, ejecútalas una a una para comprobar tus respuestas.

**Salida esperada** (los errores se muestran con su mensaje de `TypeError`)

#### Ejercicio 6.7 · Repartir ficheros entre ejecutores

Un job tiene que procesar varios ficheros de tamaños muy distintos y dispone de varios ejecutores. Para equilibrar la carga se usa una regla sencilla: se ordenan los ficheros de mayor a menor y cada uno se asigna al ejecutor que **menos megas lleva acumulados** en ese momento. Si hay empate, se elige el de número más bajo.
1. Escribe `asignar_tareas(ficheros, num_ejecutores=3)`, que devuelva un diccionario `"ejecutor_N" → {"ficheros": [...], "mb": total}`.
2. Escribe `desequilibrio(asignacion)`, que devuelva la carga máxima dividida entre la mínima, con dos decimales.
3. Muestra el reparto con 3 ejecutores y compara su desequilibrio con el de 4 ejecutores.

**Salida esperada**

In [0]:
from operator import itemgetter

ficheros = [
    ("f1.parquet", 512),
    ("f2.parquet", 128),
    ("f3.parquet", 900),
    ("f4.parquet", 256),
    ("f5.parquet", 640),
    ("f6.parquet", 300),
    ("f7.parquet", 128),
]


def asignar_tareas(ficheros, num_ejecutores=3):
    asignacion = {
        f"ejecutor_{i}": {"ficheros": [], "mb": 0} for i in range(num_ejecutores)
    }
    ordenados = sorted(ficheros, key=itemgetter(1), reverse=True)

    for nombre, tamano in ordenados:
        peor_ejecutor = min(
            asignacion.keys(), key=lambda e: (asignacion[e]["mb"], int(e.split("_")[1]))
        )
        asignacion[peor_ejecutor]["ficheros"].append(nombre)
        asignacion[peor_ejecutor]["mb"] += tamano

    return asignacion


def desequilibrio(asignacion):
    cargas = [datos["mb"] for datos in asignacion.values()]
    return round(max(cargas) / min(cargas), 2)


# Reparto con 3 ejecutores
asig_3 = asignar_tareas(ficheros, num_ejecutores=3)
for ejecutor, datos in asig_3.items():
    print(f"{ejecutor}: {datos['mb']:>4} MB -> {datos['ficheros']}")
print("Desequilibrio (máx / mín):", desequilibrio(asig_3))

# Comparativa con 4 ejecutores
asig_4 = asignar_tareas(ficheros, num_ejecutores=4)
cargas_4 = [datos["mb"] for datos in asig_4.values()]
print(
    f"Con 4 ejecutores: {cargas_4} -> desequilibrio {desequilibrio(asig_4)}"
)

ejecutor_0:  900 MB -> ['f3.parquet']
ejecutor_1: 1024 MB -> ['f5.parquet', 'f4.parquet', 'f7.parquet']
ejecutor_2:  940 MB -> ['f1.parquet', 'f6.parquet', 'f2.parquet']
Desequilibrio (máx / mín): 1.14
Con 4 ejecutores: [900, 640, 640, 684] -> desequilibrio 1.41


#### Ejercicio 6.8 · Generar el comando `spark-submit`

Para lanzar un job de Spark en un clúster se usa el comando `spark-submit`, con opciones como el gestor de recursos (`--master`), el modo de despliegue o el número de ejecutores, y parámetros de configuración `--conf clave=valor`.

Escribe `construir_spark_submit(script, *args_app, master="yarn", deploy_mode="cluster", num_executors=2, **conf)`. La función devuelve el comando como texto, con este orden: opciones fijas, opciones `--conf`, script y argumentos de la aplicación. En los nombres de `conf`, los `_` se sustituyen por `.` (`spark_executor_memory` → `spark.executor.memory`). Comprueba la función con estas llamadas:

In [0]:
def construir_spark_submit(
    script,
    *args_app,
    master="yarn",
    deploy_mode="cluster",
    num_executors=2,
    **conf,
):
    partes = [
        "spark-submit",
        f"--master {master}",
        f"--deploy-mode {deploy_mode}",
        f"--num-executors {num_executors}",
    ]

    for clave, valor in conf.items():
        clave_spark = clave.replace("_", ".")
        partes.append(f"--conf {clave_spark}={valor}")

    partes.append(script)

    if args_app:
        partes.extend(str(a) for a in args_app)

    return " ".join(partes)


print(construir_spark_submit("carga_ventas.py"))
print(
    construir_spark_submit(
        "carga_ventas.py",
        "--fecha",
        "2026-10-15",
        num_executors=10,
        spark_executor_memory="8g",
        spark_sql_shuffle_partitions=400,
    )
)
print(construir_spark_submit("prueba.py", master="local[4]", deploy_mode="client"))

spark-submit --master yarn --deploy-mode cluster --num-executors 2 carga_ventas.py
spark-submit --master yarn --deploy-mode cluster --num-executors 10 --conf spark.executor.memory=8g --conf spark.sql.shuffle.partitions=400 carga_ventas.py --fecha 2026-10-15
spark-submit --master local[4] --deploy-mode client --num-executors 2 prueba.py


#### Ejercicio 6.9 · Controlar el gasto de una aplicación con LLM

Los proveedores de modelos de lenguaje cobran por **tokens**, con un precio distinto para los de entrada (el prompt) y los de salida (la respuesta).
1. Escribe `coste_peticion(modelo, tokens_entrada, tokens_salida)`, que devuelva el coste en USD de una petición. Pruébala con la primera petición desempaquetando la tupla con .
2. Escribe `resumen_costes(peticiones, presupuesto_diario=0.05)`, que devuelva tres valores: un diccionario por modelo con su número de peticiones y su coste, el coste total y si se ha superado el presupuesto.
3. Escribe `proyectar(coste_diario, dias=30)`, que devuelva el coste estimado para ese número de días, con dos decimales.

In [0]:
TARIFAS = {"modelo_ligero": (0.25, 1.25), "modelo_avanzado": (3.00, 15.00)}

peticiones = [
    ("modelo_ligero", 1200, 300),
    ("modelo_avanzado", 5400, 900),
    ("modelo_ligero", 800, 150),
    ("modelo_avanzado", 12000, 2000),
    ("modelo_ligero", 2500, 600),
]


def coste_peticion(modelo, tokens_entrada, tokens_salida):
    tarifa_in, tarifa_out = TARIFAS[modelo]
    return (tokens_entrada * tarifa_in + tokens_salida * tarifa_out) / 1_000_000


def resumen_costes(peticiones, presupuesto_diario=0.05):
    resumen = {}
    coste_total = 0.0

    for modelo, tin, tout in peticiones:
        if modelo not in resumen:
            resumen[modelo] = {"peticiones": 0, "coste": 0.0}
        coste = coste_peticion(modelo, tin, tout)
        resumen[modelo]["peticiones"] += 1
        resumen[modelo]["coste"] += coste
        coste_total += coste

    supera = coste_total > presupuesto_diario
    return resumen, coste_total, supera


def proyectar(coste_diario, dias=30):
    return round(coste_diario * dias, 2)


# Comprobaciones
coste_primera = coste_peticion(*peticiones[0])
print(f"Coste de la primera petición: {coste_primera:.6f} USD")

resumen, total_dia, supera = resumen_costes(peticiones)
for mod, d in resumen.items():
    print(f"{mod:<16} {d['peticiones']} peticiones -> {d['coste']:.6f} USD")

print(f"Total del día: {total_dia:.6f} USD | ¿Supera el presupuesto? {supera}")
print(f"Proyección a 30 días: {proyectar(total_dia, 30)} USD")
print(f"Proyección a 365 días: {proyectar(total_dia, 365)} USD")

Coste de la primera petición: 0.000675 USD
modelo_ligero    3 peticiones -> 0.002437 USD
modelo_avanzado  2 peticiones -> 0.095700 USD
Total del día: 0.098138 USD | ¿Supera el presupuesto? True
Proyección a 30 días: 2.94 USD
Proyección a 365 días: 35.82 USD


#### Ejercicio 6.10 · Validar la salida estructurada de un LLM

Un LLM clasifica los tickets de soporte y devuelve un diccionario por ticket. Antes de guardar sus respuestas en la base de datos hay que comprobarlas: los modelos a veces omiten campos o inventan valores que no existen.
1. Escribe `validar_respuesta(respuesta, obligatorios=("categoria", "prioridad", "resumen"), **permitidos)`, que devuelva `(es_valida, errores)`. Un campo obligatorio falla si no existe o está vacío. Cada argumento de `permitidos` indica los valores aceptados de un campo, como `categoria=CATEGORIAS`.
2. Escribe `procesar_respuestas(respuestas, **permitidos)`, que separe los tickets aceptados de los que necesitan revisión humana. Debe pasar `permitidos` a `validar_respuesta()` con `*`.
3. Muestra los resultados y la tasa de respuestas válidas.

In [0]:
CATEGORIAS = {"facturacion", "tecnico", "envio"}
PRIORIDADES = {"baja", "media", "alta"}

respuestas = [
    {
        "ticket": "T1",
        "categoria": "envio",
        "prioridad": "alta",
        "resumen": "Pedido no entregado",
    },
    {
        "ticket": "T2",
        "categoria": "Facturación",
        "prioridad": "media",
        "resumen": "Cobro duplicado",
    },
    {"ticket": "T3", "categoria": "tecnico", "resumen": "Error al iniciar sesión"},
    {"ticket": "T4", "categoria": "tecnico", "prioridad": "urgente", "resumen": ""},
    {
        "ticket": "T5",
        "categoria": "facturacion",
        "prioridad": "baja",
        "resumen": "Pide factura",
    },
]


def validar_respuesta(
    respuesta, obligatorios=("categoria", "prioridad", "resumen"), **permitidos
):
    errores = []
    for campo in obligatorios:
        valor = respuesta.get(campo)
        if valor is None or valor == "":
            errores.append(f"falta '{campo}'")

    for campo, validos in permitidos.items():
        if campo in respuesta and respuesta[campo]:
            if respuesta[campo] not in validos:
                errores.append(f"'{campo}' no válido: {respuesta[campo]}")

    es_valida = len(errores) == 0
    return es_valida, errores


def procesar_respuestas(respuestas, **permitidos):
    aceptadas = []
    revision = {}

    for r in respuestas:
        valida, errores = validar_respuesta(r, **permitidos)
        if valida:
            aceptadas.append(r["ticket"])
        else:
            revision[r["ticket"]] = errores

    return aceptadas, revision


aceptadas, revision = procesar_respuestas(
    respuestas, categoria=CATEGORIAS, prioridad=PRIORIDADES
)

print(f"Aceptadas: {aceptadas}")
print("A revisión humana:")
for ticket, errs in revision.items():
    print(f"  {ticket}: {errs}")

tasa = (len(aceptadas) / len(respuestas)) * 100
print(f"Tasa de respuestas válidas: {tasa:.0f} %")

Aceptadas: ['T1', 'T5']
A revisión humana:
  T2: ["'categoria' no válido: Facturación"]
  T3: ["falta 'prioridad'"]
  T4: ["falta 'resumen'", "'prioridad' no válido: urgente"]
Tasa de respuestas válidas: 40 %
